# Project 1: E-commerce Delivery & Logistics Performance Analysis
## Week 2: Master Segment Comparison & Delay Pattern Analysis
**Lead Analyst**: Aman Shakil  
**Deliverable**: Master Cross-Functional Segment Comparison Notebook

This notebook synthesizes Week 2 findings across all operational dimensions:
1. **Executive Scorecard**: Baseline OTD, Cancellation, Processing, Transit, and Satisfaction metrics.
2. **Carrier & Dispatch Latency**: Warehouse handoff vs. carrier road transit duration.
3. **Regional Corridors**: Intrastate vs. Interstate fulfillment, state-by-state delay disparities.
4. **Temporal Patterns**: Weekday ordering effects and weekend processing lag.
5. **Product Categories**: Category delivery durations, freight weights, and cancellation rates.
6. **Customer Rating Degradation Curves**: Empirical review score drop relative to delay days.


In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 25)

# Load validated KPI dataset
kpi_df = pd.read_csv('../data/processed/validated_kpi_dataset.csv')
valid = kpi_df[kpi_df['is_eligible_otd'] == 1].copy()
print(f"Loaded {len(kpi_df):,} placed orders ({len(valid):,} valid delivered orders).")


## 1. Executive Logistics KPI Scorecard
Synthesizing the core high-level metrics established in `reports/kpi_definitions.md`.


In [ ]:
from sys import path
path.append('../src')
from kpis import compute_executive_dashboard_kpis

exec_kpis = compute_executive_dashboard_kpis(kpi_df)
scorecard = pd.DataFrame(list(exec_kpis.items()), columns=['KPI Name', 'Calculated Value'])
display(scorecard)


## 2. Order Processing vs. Carrier Transit Duration (Warehouse vs. Road)
Evaluating where delivery time is spent: in the merchant warehouse (processing) or in the carrier transit network.


In [ ]:
print(f"Mean Order Processing Time: {valid['order_processing_time_days'].mean():.2f} days (Median: {valid['order_processing_time_days'].median():.2f} days)")
print(f"Mean Carrier Transit Time:  {valid['carrier_transit_time_days'].mean():.2f} days (Median: {valid['carrier_transit_time_days'].median():.2f} days)")
print(f"Mean Total Delivery Time:   {valid['total_delivery_duration_days'].mean():.2f} days (Median: {valid['total_delivery_duration_days'].median():.2f} days)")

fig, ax = plt.subplots(figsize=(10, 4))
sns.kdeplot(valid['order_processing_time_days'].clip(0, 15), label='Processing (Warehouse)', color='#ff7f0e', fill=True, alpha=0.3)
sns.kdeplot(valid['carrier_transit_time_days'].clip(0, 35), label='Transit (Carrier)', color='#1f77b4', fill=True, alpha=0.3)
plt.title('Distribution Comparison: Warehouse Processing vs. Carrier Transit Days', fontweight='bold')
plt.xlabel('Days')
plt.legend()
plt.tight_layout()
plt.show()


## 3. Regional Disparities: Intrastate vs. Interstate Shipping Corridors


In [ ]:
route_comp = valid.groupby('is_interstate').agg(
    order_volume=('order_id', 'count'),
    otd_rate_pct=('delivery_outcome', lambda x: (x == 'on_time').mean() * 100),
    median_days=('total_delivery_duration_days', 'median'),
    mean_days=('total_delivery_duration_days', 'mean'),
    avg_freight=('total_freight_value', 'mean'),
    avg_rating=('review_score', 'mean')
)
route_comp.index = ['Intrastate (Same State)', 'Interstate (Cross-State)']
display(route_comp.round(2))


## 4. Temporal Patterns: Day of Week Order Latency


In [ ]:
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_comp = valid.groupby('purchase_weekday').agg(
    order_count=('order_id', 'count'),
    avg_processing_days=('order_processing_time_days', 'mean'),
    median_total_days=('total_delivery_duration_days', 'median'),
    delay_rate_pct=('delivery_outcome', lambda x: (x == 'delayed').mean() * 100)
).reindex(days_order)

display(weekday_comp.round(2))


## 5. Product Category Logistics Complexity


In [ ]:
cat_perf = valid.groupby('primary_category').agg(
    orders=('order_id', 'count'),
    median_delivery_days=('total_delivery_duration_days', 'median'),
    delay_rate_pct=('delivery_outcome', lambda x: (x == 'delayed').mean() * 100),
    avg_freight=('total_freight_value', 'mean'),
    avg_rating=('review_score', 'mean')
).query('orders >= 100').sort_values('delay_rate_pct', ascending=False)

print("Top 10 Highest-Delay Product Categories (min 100 orders):")
display(cat_perf.head(10).round(2))


## 6. Customer Review Score Decay Curves
Empirical relationship between delivery delay severity and customer satisfaction drop.


In [ ]:
decay = valid.dropna(subset=['review_score']).groupby('delay_severity_bucket').agg(
    order_volume=('order_id', 'count'),
    avg_review_score=('review_score', 'mean'),
    pct_1star=('review_score', lambda x: (x <= 1.5).mean() * 100),
    pct_5star=('review_score', lambda x: (x >= 4.5).mean() * 100)
)
display(decay.round(2))
